# 📊 Modul 04 Hands-on: Model Evaluation & Selection

Pada hands-on ini, Anda akan membongkar jebakan evaluasi model:
1. **The Sabotage Lab: The Accuracy Trap!** Model fraud 99.5% akurat yang ternyata tidak berguna.
2. **Confusion Matrix, Precision, Recall, & F1-Score**.
3. **ROC-AUC Curve & Stratified Cross Validation**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, ConfusionMatrixDisplay

# Buat dataset imbalanced parah (99% normal, 1% fraud)
X, y = make_classification(n_samples=2000, n_features=10, weights=[0.99, 0.01], random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print(f"Distribusi kelas di data test: Normal={sum(y_test==0)}, Fraud={sum(y_test==1)}")

## 1. The Sabotage Lab: The Accuracy Trap
Kita bandingkan model 'bodoh' (selalu menebak kelas mayoritas) dengan model terkalibrasi.

In [ ]:
# Model Dummy yang selalu tebak kelas mayoritas (Normal)
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
dummy_acc = dummy.score(X_test, y_test)

# Model Logistic Regression dengan penyeimbang bobot kelas (class_weight="balanced")
real_model = LogisticRegression(class_weight="balanced").fit(X_train, y_train)
real_acc = real_model.score(X_test, y_test)

print(f"Akurasi Model Dummy (Bodoh)     : {dummy_acc*100:.2f}% (Tinggi Palsu!)")
print(f"Akurasi Logistic (Class Balanced): {real_acc*100:.2f}%")

print("\n--- EVALUASI SEBENARNYA (CLASSIFICATION REPORT) ---")
print("Report Model Dummy (Recall Fraud = 0!):\n", classification_report(y_test, dummy.predict(X_test), zero_division=0))
print("Report Model Balanced:\n", classification_report(y_test, real_model.predict(X_test)))

## 2. Visualisasi Confusion Matrix & ROC-AUC Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# 1. Confusion Matrix
cm = confusion_matrix(y_test, real_model.predict(X_test))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Normal", "Fraud"])
disp.plot(ax=axes[0], cmap="Blues")
axes[0].set_title("Confusion Matrix (Balanced)")

# 2. ROC Curve
probs = real_model.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, probs)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC Curve (AUC = {roc_auc:.3f})")
axes[1].plot([0, 1], [0, 1], color="navy", lw=1.5, linestyle="--")
axes[1].set_xlabel("False Positive Rate"); axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("Receiver Operating Characteristic (ROC)")
axes[1].legend(loc="lower right"); axes[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout(); plt.show()